# PlanForge
> Your AI business advisor. Tell it what you want to achieve, have a conversation, get a plan.

*v0.5.0*

In [ ]:
# @title Setup
# @markdown **Run this cell every session.**
!pip install -q pysolvr-client
import sys, time as _time
from google.colab import userdata, drive
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

try:
    API_KEY = userdata.get('PLANFORGE_API_KEY')
except userdata.SecretNotFoundError:
    display(HTML('<div style="background:#1e293b;border:1px solid #ef4444;border-radius:8px;padding:16px;font-family:Inter,system-ui,sans-serif;color:#f1f5f9"><b style="color:#ef4444">API key not found</b><ol style="color:#94a3b8;font-size:13px;margin-top:8px"><li>Click key icon in left sidebar</li><li>Add: <code>PLANFORGE_API_KEY</code></li><li>Toggle Notebook access ON</li><li>Re-run this cell</li></ol></div>'))
    assert False, 'PLANFORGE_API_KEY not found'

drive.mount('/content/drive', force_remount=False)
sys.path.insert(0, '/content')

from pysolvr_client import ApiClient, Display, DriveManager

API_BASE = 'https://ynmwpf8duf.execute-api.us-east-1.amazonaws.com/dev'
client = ApiClient(API_BASE, API_KEY)
ui = Display('#3B82F6', '#10B981')
drive_mgr = DriveManager('planforge', ['profiles', 'plans', 'exports', 'uploads'])
drive_mgr.ensure_folders()

# Session state (shared across cells)
_state = {'goal': None, 'goal_info': {}, 'topics': [], 'current_topic': None, 'history': [], 'locked': {}}

if client.health_check():
    ui.success('Connected', f'Drive: {drive_mgr.root}')
else:
    ui.error('Connection failed', 'Check API key')


In [ ]:
# @title * What do you want to achieve? (required)
GOAL = 'I need funding for my startup'  # @param ["I need funding for my startup", "I need a UK visa for my business", "I want to validate my idea", "I need a bank loan", "I'm applying to an accelerator", "I need a grant (Innovate UK)", "I just want a quick viability check"]

_goal_map = {
    'I need funding for my startup': 'seed_pitch',
    'I need a UK visa for my business': 'innovator_visa',
    'I want to validate my idea': 'validate',
    'I need a bank loan': 'bank_loan',
    "I'm applying to an accelerator": 'accelerator',
    'I need a grant (Innovate UK)': 'innovate_uk',
    'I just want a quick viability check': 'scorecard',
}
_state['goal'] = _goal_map.get(GOAL, 'scorecard')
_state['history'] = []
_state['locked'] = {}

result = client.call('GET', '/goals')
if result['ok']:
    goals = {g['id']: g for g in result['data']['goals']}
    goal_info = goals.get(_state['goal'], {})
    _state['goal_info'] = goal_info
    _state['topics'] = [{'id': t['id'], 'name': t['name']} for t in goal_info.get('required_topics', [])]
    _state['current_topic'] = _state['topics'][0]['id'] if _state['topics'] else None
    names = [t['name'] for t in _state['topics']]
    ui.card(f'Goal: {goal_info.get("name", GOAL)}',
            f'<b>Audience:</b> {goal_info.get("audience", "you")}<br>'
            f'<b>Topics to cover:</b> {", ".join(names)}<br><br>'
            f'Run the next cell to start your discovery conversation.',
            status='success')
else:
    ui.success(f'Goal set: {GOAL}', 'Run the next cell to start chatting')


In [ ]:
# @title Discovery Chat

if not _state.get('goal') or not _state.get('topics'):
    ui.card('No goal selected', 'Run the <b>goal selection</b> cell above first.', status='error')
    assert False, 'Goal not set'

display(HTML('<p style="color:#94a3b8;font-family:Inter,sans-serif;font-size:13px;margin:0 0 8px 0">'
             'Tell me about your business. Type a message or paste text below.</p>'))

# --- Widgets ---
_progress_out = widgets.Output()
_chat_out = widgets.Output(layout={'border': '1px solid #334155', 'border_radius': '12px',
                                    'padding': '16px', 'max_height': '400px',
                                    'overflow_y': 'auto'})
_input_area = widgets.Textarea(placeholder='Type your message or paste text (CV, financials, notes)...',
                               layout=widgets.Layout(width='100%', height='80px'))
_send_btn = widgets.Button(description='Send', button_style='primary',
                            layout=widgets.Layout(width='80px', height='36px'))
_lock_btn = widgets.Button(description='\u2713 Lock in', button_style='success',
                            layout=widgets.Layout(width='100px', height='36px'))
_lock_btn.layout.display = 'none'
_status_out = widgets.Output()


def _render_progress():
    with _progress_out:
        clear_output(wait=True)
        if not _state['topics']:
            return
        done = sum(1 for t in _state['topics'] if t['id'] in _state['locked'])
        total = len(_state['topics'])
        pct = int(done / total * 100) if total else 0
        html = f'<div style="background:#1e293b;border-radius:8px;padding:12px;font-family:Inter,sans-serif;margin-bottom:8px">'
        html += f'<div style="font-size:12px;color:#94a3b8;margin-bottom:6px">{done}/{total} topics covered</div>'
        html += f'<div style="background:#334155;border-radius:4px;height:6px;margin-bottom:10px"><div style="background:#10b981;height:6px;border-radius:4px;width:{pct}%"></div></div>'
        for t in _state['topics']:
            tid, name = t['id'], t['name']
            if tid in _state['locked']:
                style = 'background:#065f46;color:#6ee7b7;border:1px solid #10b981'
                icon = '\u2713 '
            elif tid == _state['current_topic']:
                style = 'background:#1e40af;color:#93c5fd;border:1px solid #3b82f6'
                icon = '\u25cf '
            else:
                style = 'background:#1e293b;color:#64748b;border:1px solid #334155'
                icon = ''
            html += f'<span style="{style};padding:4px 10px;border-radius:12px;font-size:11px;margin:2px 4px 2px 0;display:inline-block">{icon}{name}</span>'
        if done == total:
            html += '<div style="color:#10b981;font-size:13px;margin-top:8px;font-weight:600">\u2705 Ready to generate your plan!</div>'
        html += '</div>'
        display(HTML(html))


def _render_chat():
    with _chat_out:
        clear_output(wait=True)
        if not _state['history']:
            return
        html = ''
        for msg in _state['history']:
            if msg['role'] == 'user':
                text = msg['content'][:500] + ('...' if len(msg['content']) > 500 else '')
                html += (f'<div style="text-align:right;margin:6px 0">'
                         f'<span style="background:#1e40af;color:#f1f5f9;padding:8px 12px;'
                         f'border-radius:14px 14px 4px 14px;display:inline-block;max-width:80%;'
                         f'text-align:left;font-size:13px;font-family:Inter,sans-serif;'
                         f'white-space:pre-wrap">{text}</span></div>')
            else:
                html += (f'<div style="text-align:left;margin:6px 0">'
                         f'<span style="background:#1e293b;color:#e2e8f0;padding:8px 12px;'
                         f'border-radius:14px 14px 14px 4px;display:inline-block;max-width:80%;'
                         f'text-align:left;font-size:13px;font-family:Inter,sans-serif;'
                         f'white-space:pre-wrap">{msg["content"]}</span></div>')
        display(HTML(html))


def _switch_topic(btn):
    topic_id = btn._topic_id
    if topic_id == _state['current_topic']:
        return
    new_name = next((t['name'] for t in _state['topics'] if t['id'] == topic_id), '')
    _state['current_topic'] = topic_id
    _lock_btn.layout.display = 'none'
    _render_progress()
    with _status_out:
        clear_output(wait=True)
        display(HTML(f'<span style="color:#93c5fd;font-size:12px">Topic: {new_name}</span>'))


_topic_btns = []
for _t in _state['topics']:
    _b = widgets.Button(description=_t['name'], layout=widgets.Layout(height='28px', margin='2px'))
    _b._topic_id = _t['id']
    _b.on_click(_switch_topic)
    _topic_btns.append(_b)


def _on_send(btn):
    message = _input_area.value.strip()
    if not message:
        with _status_out:
            clear_output(wait=True)
            display(HTML('<span style="color:#f87171;font-size:12px">Type a message</span>'))
        return
    _state['history'].append({'role': 'user', 'content': message})
    _input_area.value = ''
    _lock_btn.layout.display = 'none'
    _render_chat()
    with _status_out:
        clear_output(wait=True)
        display(HTML('<span style="color:#94a3b8;font-size:12px">Thinking...</span>'))
    topic = _state['current_topic'] or 'business_overview'
    body = {'topic': topic, 'message': message, 'goal_type': _state['goal'],
            'conversation_history': [{'role': m['role'], 'content': m['content'][:1000]}
                                     for m in _state['history'][-6:] if m.get('type') != 'divider'],
            'locked_topics': _state['locked']}
    submit = client.call('POST', '/chat', body)
    if not submit['ok']:
        _state['history'].append({'role': 'assistant', 'content': f"Error: {submit.get('error', 'Failed')}"})
        with _status_out:
            clear_output(wait=True)
        _render_chat()
        return
    job_id = submit['data'].get('job_id')
    if not job_id:
        reply = submit['data'].get('response', '')
        _state['history'].append({'role': 'assistant', 'content': reply})
        with _status_out:
            clear_output(wait=True)
        _render_chat()
        return
    for _ in range(60):
        _time.sleep(2)
        poll = client.call('GET', f'/jobs/{job_id}')
        if not poll['ok']:
            break
        if poll['data'].get('status') == 'complete':
            rd = poll['data'].get('result', {})
            _state['history'].append({'role': 'assistant', 'content': rd.get('response', '')})
            if rd.get('topic_status') in ('suggested_lock', 'locked'):
                _lock_btn.layout.display = 'block'
                if rd.get('topic_status') == 'locked':
                    _do_lock()
            with _status_out:
                clear_output(wait=True)
            _render_chat()
            return
        elif poll['data'].get('status') == 'failed':
            _state['history'].append({'role': 'assistant', 'content': f"Error: {poll['data'].get('error', 'Failed')}"})
            break
    with _status_out:
        clear_output(wait=True)
    _render_chat()


def _do_lock(btn=None):
    topic = _state['current_topic']
    if not topic:
        return
    client.call('POST', '/chat', {'topic': topic, 'message': 'lock_in', 'goal_type': _state['goal'],
                                   'conversation_history': [], 'locked_topics': _state['locked']})
    _state['locked'][topic] = 'locked'
    _lock_btn.layout.display = 'none'
    for t in _state['topics']:
        if t['id'] not in _state['locked']:
            _state['current_topic'] = t['id']
            with _status_out:
                clear_output(wait=True)
                display(HTML(f'<span style="color:#10b981;font-size:12px">Locked! Next: {t["name"]}</span>'))
            break
    _render_progress()
    _render_chat()


_send_btn.on_click(_on_send)
_lock_btn.on_click(_do_lock)
_render_progress()
_render_chat()
display(widgets.VBox([
    _progress_out,
    widgets.HBox(_topic_btns, layout=widgets.Layout(flex_flow='row wrap', margin='0 0 8px 0')),
    _chat_out,
    widgets.HBox([_input_area, widgets.VBox([_send_btn, _lock_btn])]),
    _status_out,
]))


In [ ]:
# @title Generate my plan
result = client.run_async('/generate', {'goal_type': _state['goal']})
if result['ok']:
    data = result['data']
    plan_id = data.get('plan_id', '')
    content = data.get('content', data.get('plan', ''))
    ui.card('Your Plan', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{content}</pre>', status='success')
    if plan_id:
        filename = f'{_state["goal"]}_{plan_id[:8]}.md'
        drive_mgr.save('plans', filename, content, meta={'plan_id': plan_id, 'goal': _state['goal']})
        ui.success('Saved to Drive', f'plans/{filename}')
else:
    ui.error(result.get('error', 'Generation failed'), f'Status: {result.get("status")}')


In [ ]:
# @title Export to a different format
FORMAT = 'executive_summary'  # @param ["executive_summary", "pitch_deck", "one_pager", "financial_summary", "full_pdf"]

plan_files = drive_mgr.list_files('plans')
if not plan_files:
    ui.error('No plans found', 'Generate a plan first')
else:
    content = drive_mgr.load('plans', plan_files[-1])
    result = client.run_async('/export', {'content': content, 'format': FORMAT})
    if result['ok']:
        data = result['data']
        ui.card(f'Export: {FORMAT}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{data["content"]}</pre>', status='success')
        filename = f'{FORMAT}_{plan_files[-1].replace(".md", "")}.md'
        drive_mgr.save('exports', filename, data['content'], meta={'format': FORMAT})
        ui.success('Saved', f'exports/{filename}')
    else:
        ui.error(result.get('error', 'Export failed'))


In [ ]:
# @title Refine a section
SECTION = ''  # @param {type:"string"}
FEEDBACK = ''  # @param {type:"string"}

plan_files = drive_mgr.list_files('plans')
if not plan_files:
    ui.error('No plans found', 'Generate a plan first')
elif not SECTION.strip():
    ui.error('Enter a section name', 'e.g. Market Analysis, Financial Projections')
else:
    content = drive_mgr.load('plans', plan_files[-1])
    body = {'content': content, 'section': SECTION}
    if FEEDBACK.strip():
        body['feedback'] = FEEDBACK
    result = client.run_async('/refine', body)
    if result['ok']:
        data = result['data']
        ui.card(f'Refined: {data["section"]}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{data["content"]}</pre>', status='success')
    else:
        ui.error(result.get('error', 'Refinement failed'))


In [ ]:
# @title My plans
result = client.call('GET', '/plans')
if result['ok']:
    plans = result['data'].get('plans', [])
    if not plans:
        ui.card('Plans', 'No plans generated yet.')
    else:
        rows = [{'ID': p['plan_id'][:8], 'Goal': p.get('goal_type', ''), 'Created': p.get('created_at', '')[:10]} for p in plans]
        ui.table(rows)
else:
    ui.error(result.get('error', 'Could not fetch plans'))


In [ ]:
# @title Account

_tab_out = [widgets.Output() for _ in range(3)]
_tabs = widgets.Tab(children=_tab_out)
_tabs.set_title(0, 'Subscription')
_tabs.set_title(1, 'Usage')
_tabs.set_title(2, 'Support')

# Subscription tab
with _tab_out[0]:
    result = client.call('GET', '/account')
    if result['ok']:
        d = result['data']
        ui.card('Subscription', f"<b>{d.get('plan', 'Free').title()}</b><br>"
            f"Status: {d.get('status', 'active')}<br>"
            f"<a href='https://planforge.pysolvr.com/pricing' style='color:#60a5fa'>Manage plan</a>",
            status='success')
    else:
        ui.error(result.get('error', 'Could not fetch account'))

# Usage tab
with _tab_out[1]:
    result = client.get_usage()
    if result['ok']:
        data = result['data']
        ui.usage_bar(data.get('monthly_spend_usd', 0), data.get('monthly_limit_usd', 1))
    else:
        ui.error(result.get('error', 'Could not fetch usage'))

# Support tab
with _tab_out[2]:
    display(HTML('<div style="font-family:Inter,system-ui,sans-serif;color:#f1f5f9;font-size:13px;padding:8px">'
        '<p><b>Files:</b> Google Drive > pysolvr > planforge</p>'
        '<p><b>Docs:</b> <a href="https://planforge.pysolvr.com/docs" style="color:#60a5fa">https://planforge.pysolvr.com/docs</a></p>'
        '<p><b>Support:</b> <a href="mailto:support@pysolvr.com?subject=PlanForge" style="color:#60a5fa">support@pysolvr.com</a></p>'
        '</div>'))

display(_tabs)
